In [ ]:
!pip install flash-attn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import sys
import torch
import numpy as np
import random
import torch.nn as nn
import datetime
import shutil
import json
from tqdm import tqdm
from collections import defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Script Configuration ---

# --- Model & Run Parameters ---
MODEL_ID = "Qwen/Qwen3-8B"
K_SHOT = 5
BATCH_SIZE = 1
MAX_SEQ_LENGTH = 2048

# --- Hardware Simulation Parameters ---
NOISE_MEAN = 0.0
NOISE_SIGMA = 0.001
TILE_SHAPE = (32, 40) # (tile_m, block_size)

# --- ADC Simulation Parameters ---
ADC_VMIN = -0.5
ADC_VMAX = 0.5
ADC_LEVELS = 256
LSB = (ADC_VMAX - ADC_VMIN) / ADC_LEVELS

# --- ADC Offset Noise Parameters ---
ADC_OFFSET = 0.5*LSB
NOISE_MEAN += ADC_OFFSET

# --- Path Configuration ---
OUTPUT_DIR = "./drive/Shareddrives/MXPTQ/results"
CHECKPOINT_DIR = "./drive/Shareddrives/MXPTQ/checkpoints"

# --- Quantization & Noise Simulation Functions ---
def quantize_per_vector(tensor, mantissa_bits, dim):
    """Quantizes a tensor per-vector, simulating float8 storage for scales."""
    max_quant_val = (2**mantissa_bits) - 1

    max_abs_vals = torch.max(torch.abs(tensor), dim=dim, keepdim=True).values
    sf = torch.where(max_abs_vals > 0, max_abs_vals / max_quant_val, 1.0)

    # Simulate precision loss from storing SF in float8
    sf = sf.to(torch.float8_e4m3fn).to(tensor.dtype)
    sf.clamp_(min=1e-12)

    quantized_tensor = torch.round(tensor / sf)
    return quantized_tensor, sf


def add_adc_noise(tensor, mean, sigma, lsb):
    """Adds simulated ADC noise."""
    if sigma <= 0:
        return tensor
    noise_in_volts = torch.randn(tensor.size(), dtype=tensor.dtype, device=tensor.device)
    noise_in_volts.mul_(sigma).add_(mean)
    integer_noise = torch.round_(noise_in_volts.div_(lsb))
    return tensor.add_(integer_noise)

# --- Simulated Hardware Layer ---

class SimulatedFFN(nn.Module):
    """Replaces a standard FFN to simulate a tiled hardware accelerator."""
    def __init__(self, original_mlp, tile_shape, noise_mean, noise_sigma,
                 adc_vmin, adc_vmax, adc_levels):
        super().__init__()
        self.gate_proj_weight = nn.Parameter(original_mlp.gate_proj.weight)
        self.up_proj_weight = nn.Parameter(original_mlp.up_proj.weight)
        self.down_proj_weight = nn.Parameter(original_mlp.down_proj.weight)

        if original_mlp.gate_proj.bias is not None:
             self.gate_proj_bias = nn.Parameter(original_mlp.gate_proj.bias)
        else:
             self.gate_proj_bias = None

        fsr = adc_vmax - adc_vmin
        self.lsb = fsr / adc_levels
        self.noise_mean = noise_mean
        self.noise_sigma = noise_sigma
        self.tile_shape = tile_shape

    def tiled_mat_mul_vec(self, input_tensor, weight_tensor):
        """Performs the simulated tiled matrix multiplication."""
        weight_tensor = weight_tensor.T
        m, k = input_tensor.shape
        _, n = weight_tensor.shape
        tile_m, block_size = self.tile_shape
        tile_n = tile_m

        pad_m = (tile_m - m % tile_m) % tile_m
        pad_k = (block_size - k % block_size) % block_size
        pad_n = (tile_n - n % tile_n) % tile_n

        inp_pad = nn.functional.pad(input_tensor, (0, pad_k, 0, pad_m))
        w_pad = nn.functional.pad(weight_tensor, (0, pad_n, 0, pad_k))
        m_pad, k_pad = inp_pad.shape
        _, n_pad = w_pad.shape

        inp_tiles = inp_pad.unfold(0, tile_m, tile_m).unfold(1, block_size, block_size)
        w_tiles = w_pad.unfold(0, block_size, block_size).unfold(1, tile_n, tile_n)

        # Quantize per-row (dim=3)
        inp_quant_tiles, inp_sf_tiles = quantize_per_vector(inp_tiles, mantissa_bits=7, dim=3)
        # Quantize per-column (dim=2)
        w_quant_tiles, w_sf_tiles = quantize_per_vector(w_tiles, mantissa_bits=2, dim=2)

        del inp_pad, w_pad, inp_tiles, w_tiles

        accum_mantissas = torch.matmul(
            inp_quant_tiles.unsqueeze(2).to(torch.bfloat16),
            w_quant_tiles.unsqueeze(0).to(torch.bfloat16)
        )
        del inp_quant_tiles, w_quant_tiles

        noisy_accum_mantissas = add_adc_noise(accum_mantissas, self.noise_mean, self.noise_sigma, self.lsb)
        del accum_mantissas

        combined_sf = inp_sf_tiles.unsqueeze(2) * w_sf_tiles.unsqueeze(0)
        torch.mul(noisy_accum_mantissas, combined_sf, out=noisy_accum_mantissas)
        dequantized_tiles = noisy_accum_mantissas
        del combined_sf, inp_sf_tiles, w_sf_tiles, noisy_accum_mantissas

        output_tiles = torch.sum(dequantized_tiles, dim=1).to(input_tensor.dtype)
        del dequantized_tiles

        out_pad = output_tiles.permute(0, 2, 1, 3).contiguous().view(m_pad, n_pad)
        return out_pad[:m, :n]

    def forward(self, x):
        """Optimized forward pass for the simulated FFN."""
        batch_size, seq_len, hidden_dim = x.shape
        x_2d = x.view(-1, hidden_dim)

        gated_output = nn.functional.linear(x_2d, self.gate_proj_weight, self.gate_proj_bias)
        gated_states = nn.functional.silu(gated_output)

        up_states = self.tiled_mat_mul_vec(x_2d, self.up_proj_weight)

        activated_states = torch.mul(gated_states, up_states, out=gated_states)

        output_states_2d = self.tiled_mat_mul_vec(activated_states, self.down_proj_weight)

        return output_states_2d.view(batch_size, seq_len, -1)

def replace_ffn_with_simulation(model, noise_mean, noise_sigma):
    """Recursively replaces all FFN (MLP) layers with the SimulatedFFN."""
    for layer in model.model.layers:
        original_mlp = layer.mlp
        layer.mlp = SimulatedFFN(
            original_mlp,
            tile_shape=TILE_SHAPE,
            noise_mean=noise_mean,
            noise_sigma=noise_sigma,
            adc_vmin=ADC_VMIN,
            adc_vmax=ADC_VMAX,
            adc_levels=ADC_LEVELS
        )
    return model

# --- MMLU Evaluation Logic ---

def format_mmlu_prompt(dev_samples, test_sample, subject_name, k_shot):
    """Formats the classic f-string prompt for base model evaluation."""
    prompt = f"The following are multiple choice questions (with answers) about {subject_name}.\n\n"

    if k_shot > 0:
        for i in range(k_shot):
            question = dev_samples[i]['question']
            choices_str = "\n".join([f"{chr(65+j)}. {dev_samples[i]['choices'][j]}" for j in range(len(dev_samples[i]['choices']))])
            prompt += f"Question: {question}\n{choices_str}\nAnswer: {chr(65+dev_samples[i]['answer'])}\n\n"

    question = test_sample['question']
    choices_str = "\n".join([f"{chr(65+j)}. {test_sample['choices'][j]}" for j in range(len(test_sample['choices']))])
    prompt += f"Question: {question}\n{choices_str}\nAnswer:"
    return prompt, chr(65 + test_sample['answer'])

@torch.no_grad()
def evaluate_mmlu(model, tokenizer, device, checkpoint_dir, k_shot=5, batch_size=1):
    """Runs the MMLU evaluation on a single GPU."""
    print("Preparing and balancing the MMLU dataset...")
    mmlu_dataset = load_dataset("cais/mmlu", "all")
    dev_examples_by_subject = defaultdict(list)
    if k_shot > 0:
        for example in mmlu_dataset["dev"]:
            dev_examples_by_subject[example["subject"]].append(example)

    all_test_examples = []
    for subject_name in sorted(list(set(mmlu_dataset["test"]["subject"]))):
        subject_test_set = mmlu_dataset["test"].filter(lambda ex: ex["subject"] == subject_name, num_proc=4)
        subject_dev_samples = dev_examples_by_subject.get(subject_name)
        for test_sample in subject_test_set:
            # Use the classic f-string prompt for base models
            formatted_prompt, correct_answer = format_mmlu_prompt(
                subject_dev_samples, test_sample, subject_name.replace("_", " "), k_shot=k_shot
            )
            all_test_examples.append({
                "prompt": formatted_prompt,
                "choices": test_sample['choices'],
                "answer": correct_answer
            })
    random.shuffle(all_test_examples)

    total_correct = 0
    start_index = 0
    total_samples = len(all_test_examples)

    checkpoint_path = os.path.join(checkpoint_dir, "mmlu_checkpoint.json")
    if os.path.exists(checkpoint_path):
        print(f"Resuming from checkpoint {checkpoint_path}")
        with open(checkpoint_path, 'r') as f:
            checkpoint_data = json.load(f)
            start_index = checkpoint_data.get('progress', 0)
            total_correct = checkpoint_data.get('correct_count', 0)

    pbar = tqdm(range(start_index, total_samples, batch_size),
                desc="Evaluating MMLU",
                initial=start_index,
                total=total_samples)

    for i in pbar:
        batch_items = all_test_examples[i:i+batch_size]
        if not batch_items:
            continue

        num_in_batch = len(batch_items)
        prompts = [item['prompt'] for item in batch_items]
        correct_answers = [item['answer'] for item in batch_items]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding="longest"
        ).to(device)

        outputs = model(**inputs)
        last_token_logits = outputs.logits[:, -1, :]

        for j in range(num_in_batch):
            choices = batch_items[j]['choices']
            logits = last_token_logits[j]
            # Get token IDs for "A", "B", "C", "D" (no leading space)
            choice_tokens = [tokenizer.encode(f"{chr(65+k)}", add_special_tokens=False)[-1] for k in range(len(choices))]
            choice_logits = logits[choice_tokens]
            prediction = chr(65 + torch.argmax(choice_logits).item())

            if prediction == correct_answers[j]:
                total_correct += 1

        del inputs, outputs, last_token_logits, choice_logits

        if (i // batch_size) % 50 == 0 and i > start_index:
            with open(checkpoint_path, 'w') as f:
                json.dump({'progress': i + num_in_batch, 'correct_count': total_correct}, f)

    overall_accuracy = total_correct / total_samples if total_samples > 0 else 0
    print(f"\nOverall MMLU Accuracy ({k_shot}-shot): {overall_accuracy:.4f}")
    return {'overall_accuracy': overall_accuracy}



In [ ]:
class Logger:
    def __init__(self, filepath, stream):
        self.stream = stream
        # Open the file with line buffering (buffering=1) for real-time output
        self.log = open(filepath, 'a', buffering=1)

    def write(self, message):
        self.stream.write(message)
        self.log.write(message)

    def flush(self):
        # This flush method is needed for compatibility with some libraries.
        self.stream.flush()
        self.log.flush()

    def __del__(self):
        # Ensure the log file is closed when the object is destroyed.
        self.log.close()

In [ ]:
# --- Main Script Execution ---
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: Running on CPU. This will be extremely slow.")

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

os.makedirs(OUTPUT_DIR, exist_ok=True)
log_file_path = f"{OUTPUT_DIR}/quant_hw_output.log"

original_stdout = sys.stdout
original_stderr = sys.stderr
sys.stdout = Logger(log_file_path, original_stdout)
sys.stderr = Logger(log_file_path, original_stderr)

try:

    all_experiment_results = {}

    print("\n" + "="*80)
    print(f"  STARTING EXPERIMENT: Offset Noise = {ADC_OFFSET}")
    print(f"  Model: {MODEL_ID}, k_shot: {K_SHOT}, batch_size: {BATCH_SIZE}")
    print(f"  Device: {device}")
    print("="*80 + "\n")

    checkpoint_dir_run = os.path.join(CHECKPOINT_DIR, f"offset_{ADC_OFFSET}")
    if os.path.exists(checkpoint_dir_run):
        shutil.rmtree(checkpoint_dir_run)
    os.makedirs(checkpoint_dir_run, exist_ok=True)

    print(f"[Offset={ADC_OFFSET}] Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'left'

    print(f"[Offset={ADC_OFFSET}] Loading base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        use_cache=False,
        attn_implementation="flash_attention_2" if torch.cuda.is_available() else "sdpa",
        trust_remote_code=True
    )

    print(f"[Offset={ADC_OFFSET}] Replacing FFN layers with simulation...")
    quant_hw_model = replace_ffn_with_simulation(model, noise_mean=NOISE_MEAN, noise_sigma=NOISE_SIGMA)

    quant_hw_model.to(device)
    model = quant_hw_model

    results = evaluate_mmlu(
        model,
        tokenizer,
        device,
        checkpoint_dir_run,
        k_shot=K_SHOT,
        batch_size=BATCH_SIZE
    )

    if results:
        acc = results['overall_accuracy']
        all_experiment_results[f"offset_{ADC_OFFSET}"] = acc
        print(f"\n--- Results for Offset {ADC_OFFSET}: Overall Accuracy = {acc:.4f} ---")

    del model, quant_hw_model, tokenizer
    torch.cuda.empty_cache()

    print("\n" + "="*80)
    print(f"          OVERALL EXPERIMENT SUMMARY (ADC OFFSET: {ADC_OFFSET})")
    print("="*80 + "\n")
    for run_name, accuracy in all_experiment_results.items():
        print(f"Run: {run_name:<20} | Overall MMLU Accuracy: {accuracy:.4f}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    output_file = os.path.join(OUTPUT_DIR, f"final_results_offset_{ADC_OFFSET}.pt")
    print(f"\nSaving final summary to {output_file}")
    torch.save(all_experiment_results, output_file)

    print("\n--- Experiment Finished ---")
finally:
    # --- Restore original stdout and stderr ---
    # This block will run even if the code in the 'try' block fails.
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    print("--- Logging restored to notebook output ---")


  STARTING EXPERIMENT: Offset Noise = 0.001953125
  Model: Qwen/Qwen3-8B, k_shot: 5, batch_size: 1
  Device: cuda

[Offset=0.001953125] Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Filter (num_proc=4):   0%|          | 0/14042 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/14042 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/14042 [00:00<?, ? examples/s]

Evaluating MMLU: 100%|##########| 10/10 [00:34<00:00,  3.48s/it]


Overall MMLU Accuracy (5-shot): 0.7000

--- Results for Offset 0.001953125: Overall Accuracy = 0.7000 ---

          OVERALL EXPERIMENT SUMMARY (ADC OFFSET: 0.001953125)

Run: offset_0.001953125   | Overall MMLU Accuracy: 0.7000

Saving final summary to ./drive/Shareddrives/MXPTQ/results/final_results_offset_0.001953125.pt

--- Experiment Finished ---
--- Logging restored to notebook output ---
